In [1]:
import zipfile
import pandas as pd
from pathlib import Path
import regex
import unicodedata

In [ ]:
diretorio = '.'

In [3]:
groundtruth = f'{diretorio}/webtable_join_ground_truth.csv'
df_comparacao = pd.read_csv(groundtruth)

In [4]:
zipado = f'{diretorio}/webtables.zip'

In [5]:
def padronizar_coluna(colunas_cabecalho):
    lista_padronizada = []
    
    for texto in colunas_cabecalho:
        
        texto = texto.lower().strip() # Trasnformar para minúsculo e remover espaços no ínicio e fim
        texto = ''.join(regex.findall(r'[a-zA-ZÀ-ÿ0-9]', texto)) # Remover caracteres especiais e espaços internos
        texto_normalizado = unicodedata.normalize('NFKD', texto) # Normaliza o texto para separar acentos das letras
        texto = ''.join(c for c in texto_normalizado if not unicodedata.combining(c)) # Remove os caracteres que não são ASCII (ou seja, os acentos)
        
        lista_padronizada.append(texto)
    
    return set(lista_padronizada)

In [6]:
def comparar_arquivos(source, compare):
    source_p = padronizar_coluna(source)
    compare_p = padronizar_coluna(compare)
    
    colunas_em_ambos = source_p & compare_p
    colunas_diferentes = source_p ^ compare_p
    
    check = len(colunas_diferentes) > 0 and len(colunas_em_ambos) > 0
        
    increase = colunas_em_ambos == source_p or colunas_em_ambos == compare_p
    
    if check and not increase:
        return (colunas_em_ambos, colunas_diferentes)
    else:
        return (None, None)

In [7]:
matchs = []

with zipfile.ZipFile(zipado) as z:
    
    for i,r in df_comparacao.iterrows():
        df_source = None
        df_compare = None
        file_source = None
        file_compare = None
        
        for filename in z.namelist():        # lista todos os paths dentro do zip
            
            if filename.startswith('tables') and not filename.endswith("/"):
                caminho_path = Path(filename)
                
                if caminho_path.suffix == '.csv':
                    csv = caminho_path.stem + caminho_path.suffix
                    
                    if csv == r['query_table']:
                        with z.open(filename)  as f:
                            df_source = pd.read_csv(f,low_memory=False, nrows=1)
                            
                            if df_source.empty:
                                continue
                            
                            df_source = list(df_source.columns)
                            file_source = r['query_table']
                            
                    elif csv == r['candidate_table']:
                        with z.open(filename)  as f:
                            df_compare = pd.read_csv(f,low_memory=False, nrows=1)
                            
                            if df_compare.empty:
                                continue
                            
                            df_compare = list(df_compare.columns)
                            file_compare = r['candidate_table']
                            
                    if df_compare != None and df_source != None:
                        igual, diferente = comparar_arquivos(df_source, df_compare)
                        
                        if igual != None and diferente != None:
                            for i in igual:
                                matchs.append(
                                    {
                                        'source':file_source,
                                        'compare':file_compare,
                                        'coluna':i,
                                        'tipo':'match'
                                    }
                                )
                                
                            for d in diferente:
                                matchs.append(
                                    {
                                        'source':file_source,
                                        'compare':file_compare,
                                        'coluna':d,
                                        'tipo':'diferente'
                                    }
                                )
                            
                        break

In [8]:
if len(matchs) > 0:
    df = pd.DataFrame(matchs)
    df.to_csv('./gabarito_webtable.csv', index=False)